# Training Development

Your implementations live in `cs336_basics`. Key exports:
- **Model:** `TransformerLM`  
- **Training:** `AdamW`, `get_lr_cosine_schedule`, `crossentropy`, `get_batch`, `gradient_clipping`, `save_checkpoint`, `load_checkpoint`
- **Tokenizer:** `train_bpe`, `Tokenizer`

Run with: `uv run jupyter notebook notebooks/train_dev.ipynb`

If you change code in cs336_basics, restart the kernel to pick up changes.


In [9]:
before = set(globals())

# Imports
from cs336_basics import (
    Tokenizer, 
    TransformerLM, 
    AdamW, 
    get_lr_cosine_schedule,
    get_batch,
    crossentropy,
    save_bpe,
    load_bpe,
    train_bpe
)

from tqdm.auto import tqdm
import wandb

import os
import numpy as np
import torch

after = set(globals())
sorted(after - before)


['torch']

## Notes (generated)

**Data:** Your `data/` symlink points to the raw text files. You'll need to tokenize them and save as numpy memmap files for efficient random access during training.

**Tokenizer save/load:** When saving vocab to JSON, remember that not all byte values (0-255) are valid UTF-8. Use `latin-1` encoding which maps bytes 1-to-1 with characters.

**Perplexity:** exp(cross_entropy_loss) - the standard metric for language models.

**Checkpointing:** Your `save_checkpoint`/`load_checkpoint` functions handle model, optimizer state, and iteration number.

**Wandb:** Already in your dependencies. `wandb login` once in terminal, then use in code.


In [2]:
# test wandb

## Workflow
1. check for tokenizer info and if absent train the encoder
   1. takes input path, vocab size, special tokens
   2. outputs the vocab (int->bytes) list and the ordered merges
   3. save the outputs
2. encode the text and save to "npy"
   1. `orig_data.tofile("data.npy")`
   2. `data = np.memmap("data.npy", dtype=np.int32)` actually use `uint16`
   3. 

In [ ]:
# Setup (paths, device, config)

config = dict(
    # tokenizer
    training_file = "../data/TinyStoriesV2-GPT4-train.txt",
    vocab_size = 10000, # also model parameter
    special_tokens = ["<|endoftext|>"], # takes list, can make file later...
    tokenizer_dir = "../tokenizers/tinystories/",
    tokenized_file="../tokenized/tinystories_train.npy",

    # model # small, ~10M parameters
    d_model=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    context_length=125,
    rope_theta=10000,


   # # Medium model (~50M params) - better results, slower
   # d_model = 512,
   # num_heads = 8,
   # num_layers = 6,
   # d_ff = 2048,
   # context_length = 512,

    # optimizer
    optimizer_use_defaults = True,
    optimizer_lr = 0.,
    optimizer_betas = (0., 0.),
    optimizer_eps = 1e-08,
    optimizer_weight_decay = 0.,

    # training loop
    num_iters = 1000

    

)



In [ ]:
# device check

if torch.backends.mps.is_available():
    config["device"] = "mps"
elif torch.cuda.is_available():
    config["device"] = "cuda"
else:
    config["device"] = "cpu"

In [ ]:
# check if outpu file exists
if os.path.exists(config["tokenized_file"]) and os.path.isfile(config["tokenized_file"]):
    print(f"File {config["tokenized_file"]} exists")

else:

    # check if tokenizer files are present
    vocab_path = os.path.join(config["tokenizer_dir"], "vocab.json")
    merges_path = os.path.join(config["tokenizer_dir"], "merges.pkl")
    if os.path.exists(vocab_path) and os.path.exists(merges_path):
        print("Loading existing tokenizer...")
        vocab, merges = load_bpe(config["tokenizer_dir"])

    # Tokenizer training
    else:
        print("Training new tokenizer...")
        vocab, merges = train_bpe(config["training_file"], config["vocab_size"], config["special_tokens"])
        save_bpe(config["tokenizer_dir"], vocab, merges)

    tokenizer = Tokenizer(vocab, merges)
    with open(config["training_file"], "r", encoding="utf-8") as f:
        text = f.read()

    tokens = tokenizer.encode(text)

    # Save as contiguous array (memmap-compatible)
    arr = np.array(tokens, dtype=np.uint16)
    np.save(config["tokenized_file"], arr)

    print(f"Saved {len(tokens):,} tokens to {config["tokenized_file"]}")




Training new tokenizer...


BPE merges: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 9743/9743 [00:23<00:00, 418.37it/s]


In [4]:
# Data preparation
tokens = np.load(config["tokenized_file"], mmap_mode='r')
# no np.memmap(...)?


In [ ]:
# Model
model = TransformerLM(
    config["vocab_size"],
    config["d_model"],
    config["num_heads"],
    config["num_layers"],
    config["d_ff"],
    config["context_length"],
    config["rope_theta"]
).to(config["device"])


In [ ]:
# Optimizer
if config["optimizer_use_defaults"]:
    optimizer = AdamW(model.parameters()) # using defaults
else:
    # TODO: optimizer call with parameters
    pass


In [ ]:
# Training loop

for i in range(config["num_iters"]):
    